# Assignment 27 — Deploying Efficient Deep Learning Systems

In [1]:
# If a package is missing in your environment, uncomment and run:
#!pip install -q tensorflow scikit-learn mlflow matplotlib pandas


In [2]:
import os
import time
import random
import tempfile
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.keras

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("MLflow:", mlflow.__version__)
print("Device:", "GPU" if tf.config.list_physical_devices("GPU") else "CPU")


Python: 3.13.9
TensorFlow: 2.21.0
Keras: 3.15.0
MLflow: 3.17.0
Device: CPU


## 1. Load and prepare the Fashion Mnist image dataset


In [3]:
# Load Fashion-MNIST dataset from TensorFlow/Keras
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Convert pixel values from 0-255 to 0-1
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Keras Conv2D expects channels-last tensors: (batch, height, width, channels)
X_train_tf = X_train[..., None]
X_test_tf = X_test[..., None]

# Create TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices((X_train_tf, y_train))
train_ds = train_ds.shuffle(
    len(X_train),
    seed=SEED,
    reshuffle_each_iteration=True
).batch(64)

test_ds = tf.data.Dataset.from_tensor_slices(
    (X_test_tf, y_test)
).batch(128)

print("Train examples:", len(X_train), "| Test examples:", len(X_test))
print("Image shape:", X_train_tf.shape[1:])
print("Classes:", sorted(np.unique(y_train).tolist()))

Train examples: 60000 | Test examples: 10000
Image shape: (28, 28, 1)
Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 2. Define a teacher and a compact student

In [4]:
def SmallCNN(width=16, dropout=0.0):
    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(width, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(width * 2, 3, padding="same", activation="relu")(x)
    # With 28x28 inputs, MaxPooling2D(2) downsamples to 14x14.
    # AveragePooling2D with pool_size=7 results in the target 2x2 spatial size.
    x = layers.AveragePooling2D(pool_size=7)(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(10)(x)
    return keras.Model(inputs, outputs, name=f"SmallCNN_w{width}")


def train_supervised(model, dataset, epochs=8, lr=1e-3):
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    for epoch in range(epochs):
        total_loss = keras.metrics.Mean()
        for xb, yb in dataset:
            with tf.GradientTape() as tape:
                logits = model(xb, training=True)
                loss = loss_fn(yb, logits)
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            total_loss.update_state(loss)
        print(f"epoch {epoch+1:02d}/{epochs} loss={total_loss.result().numpy():.4f}")
    return model


def predict_logits(model, dataset):
    outputs, labels = [], []
    for xb, yb in dataset:
        outputs.append(model(xb, training=False))
        labels.append(yb)
    return tf.concat(outputs, axis=0), tf.concat(labels, axis=0)


def evaluate(model, dataset):
    logits, labels = predict_logits(model, dataset)
    preds = tf.argmax(logits, axis=1).numpy()
    truth = labels.numpy()
    return float(accuracy_score(truth, preds)), preds, truth


def count_parameters(model):
    return int(model.count_params())


def model_file_size_mb(model):
    fd, path = tempfile.mkstemp(suffix=".keras")
    os.close(fd)
    try:
        model.save(path)
        return os.path.getsize(path) / (1024 * 1024)
    finally:
        if os.path.exists(path):
            os.remove(path)


def latency_ms_per_image(model, sample_batch=X_test_tf[:1], repeats=300):
    sample = tf.convert_to_tensor(sample_batch)
    # Warm-up
    for _ in range(20):
        _ = model(sample, training=False)
    start = time.perf_counter()
    for _ in range(repeats):
        _ = model(sample, training=False)
    return (time.perf_counter() - start) * 1000 / repeats


def metrics_for(model, dataset):
    acc, _, _ = evaluate(model, dataset)
    return {
        "test_accuracy": acc,
        "parameter_count": count_parameters(model),
        "keras_model_size_mb": float(model_file_size_mb(model)),
        "latency_ms_per_image": float(latency_ms_per_image(model))
    }

## 3. Train the teacher model

In [5]:
mlflow.set_experiment("assignment27-efficient-deep-learning")

teacher = SmallCNN(width=32, dropout=0.05)
with mlflow.start_run(run_name="teacher_supervised") as run:
    mlflow.log_params({
        "model_role": "teacher",
        "architecture": "SmallCNN",
        "framework": "TensorFlow/Keras",
        "width": 32,
        "epochs": 8,
        "learning_rate": 0.001,
        "batch_size": 64,
        "seed": SEED,
        "dataset": "fashion_mnist_28x28"
    })
    teacher = train_supervised(teacher, train_ds, epochs=8, lr=1e-3)
    teacher_metrics = metrics_for(teacher, test_ds)
    mlflow.log_metrics(teacher_metrics)
    mlflow.keras.log_model(teacher, artifact_path="teacher_model")
    mlflow.set_tag("purpose", "baseline teacher")
    print("Teacher run ID:", run.info.run_id)
    print("Teacher metrics:", teacher_metrics)

2026/10/12 21:11:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/12 21:11:24 INFO mlflow.store.db.utils: Updating database tables
2026/10/12 21:11:27 INFO mlflow.tracking.fluent: Experiment with name 'assignment27-efficient-deep-learning' does not exist. Creating a new experiment.


epoch 01/8 loss=0.9087
epoch 02/8 loss=0.6045
epoch 03/8 loss=0.5363
epoch 04/8 loss=0.4932
epoch 05/8 loss=0.4602
epoch 06/8 loss=0.4355
epoch 07/8 loss=0.4182
epoch 08/8 loss=0.4025


2026/10/12 21:20:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/12 21:20:03 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


Teacher run ID: 3b777546d6f9465b83f1d29db30b1c28
Teacher metrics: {'test_accuracy': 0.8517, 'parameter_count': 21386, 'keras_model_size_mb': 0.11107063293457031, 'latency_ms_per_image': 9.291437333255695}


## 4. Train a compact student using knowledge distillation

In [6]:
def train_distilled_student(teacher, student, dataset, epochs=10, lr=1e-3,
                            temperature=3.0, alpha=0.5):
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    hard_loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    # Teacher remains frozen during distillation.
    teacher.trainable = False

    for epoch in range(epochs):
        running_loss = keras.metrics.Mean()

        for xb, yb in dataset:
            teacher_logits = teacher(xb, training=False)
            teacher_soft = tf.nn.softmax(teacher_logits / temperature, axis=1)

            with tf.GradientTape() as tape:
                student_logits = student(xb, training=True)

                hard_loss = hard_loss_fn(yb, student_logits)

                student_log_soft = tf.nn.log_softmax(
                    student_logits / temperature, axis=1
                )
                soft_loss = tf.reduce_mean(
                    tf.reduce_sum(
                        teacher_soft * (
                            tf.math.log(tf.clip_by_value(teacher_soft, 1e-7, 1.0))
                            - student_log_soft
                        ),
                        axis=1
                    )
                ) * (temperature ** 2)

                loss = alpha * hard_loss + (1 - alpha) * soft_loss

            grads = tape.gradient(loss, student.trainable_variables)
            optimizer.apply_gradients(zip(grads, student.trainable_variables))
            running_loss.update_state(loss)

        print(
            f"epoch {epoch+1:02d}/{epochs} "
            f"combined_loss={running_loss.result().numpy():.4f}"
        )
    return student


student = SmallCNN(width=8, dropout=0.0)
with mlflow.start_run(run_name="student_distillation") as run:
    mlflow.log_params({
        "model_role": "student",
        "architecture": "SmallCNN",
        "framework": "TensorFlow/Keras",
        "width": 8,
        "epochs": 10,
        "learning_rate": 0.001,
        "batch_size": 64,
        "temperature": 3.0,
        "alpha_hard_label_loss": 0.5,
        "seed": SEED,
        "dataset": "fashion_mnist_28x28"
    })
    student = train_distilled_student(
        teacher, student, train_ds, epochs=10, lr=1e-3,
        temperature=3.0, alpha=0.5
    )
    student_metrics = metrics_for(student, test_ds)
    mlflow.log_metrics(student_metrics)
    mlflow.keras.log_model(student, artifact_path="student_model")
    mlflow.set_tag("purpose", "distilled compact student")
    print("Student run ID:", run.info.run_id)
    print("Student metrics:", student_metrics)

epoch 01/10 combined_loss=1.8778
epoch 02/10 combined_loss=0.7201
epoch 03/10 combined_loss=0.6116
epoch 04/10 combined_loss=0.5549
epoch 05/10 combined_loss=0.5155
epoch 06/10 combined_loss=0.4842
epoch 07/10 combined_loss=0.4596
epoch 08/10 combined_loss=0.4405
epoch 09/10 combined_loss=0.4226
epoch 10/10 combined_loss=0.4072


2026/10/12 21:30:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/12 21:30:33 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


Student run ID: b19f40fac4b84401903372905d6509f7
Student metrics: {'test_accuracy': 0.8048, 'parameter_count': 1898, 'keras_model_size_mb': 0.03676319122314453, 'latency_ms_per_image': 8.53124666668009}


## 5. Compare accuracy and deployment cost

In [7]:
comparison = pd.DataFrame([
    {"model": "Teacher", **teacher_metrics},
    {"model": "Distilled student", **student_metrics}
])
display(comparison)

with mlflow.start_run(run_name="comparison_summary"):
    mlflow.log_param("comparison", "teacher_vs_distilled_student")
    for _, row in comparison.iterrows():
        prefix = "teacher" if row["model"] == "Teacher" else "student"
        for metric in ["test_accuracy", "parameter_count", "keras_model_size_mb", "latency_ms_per_image"]:
            mlflow.log_metric(f"{prefix}_{metric}", float(row[metric]))
    comparison_path = "assignment27_model_comparison.csv"
    comparison.to_csv(comparison_path, index=False)
    mlflow.log_artifact(comparison_path)
print("Saved comparison CSV:", comparison_path)


,model,test_accuracy,parameter_count,keras_model_size_mb,latency_ms_per_image
0,Teacher,0.8517,21386,0.111071,9.291437
1,Distilled student,0.8048,1898,0.036763,8.531247


Saved comparison CSV: assignment27_model_comparison.csv


## 6. Additional evaluation report

In [8]:
student_acc, student_preds, student_truth = evaluate(student, test_ds)
print("Student test accuracy:", round(student_acc, 4))
print(classification_report(student_truth, student_preds, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(student_truth, student_preds))


Student test accuracy: 0.8048
              precision    recall  f1-score   support

           0       0.78      0.77      0.77      1000
           1       0.99      0.95      0.97      1000
           2       0.67      0.70      0.68      1000
           3       0.70      0.90      0.79      1000
           4       0.76      0.57      0.65      1000
           5       0.94      0.94      0.94      1000
           6       0.46      0.44      0.45      1000
           7       0.90      0.92      0.91      1000
           8       0.94      0.93      0.94      1000
           9       0.94      0.93      0.93      1000

    accuracy                           0.80     10000
   macro avg       0.81      0.80      0.80     10000
weighted avg       0.81      0.80      0.80     10000

Confusion matrix:
[[765   1  13 114   6   2  84   0  15   0]
 [  1 951   6  36   4   0   2   0   0   0]
 [ 11   1 701  15  83   0 182   0   7   0]
 [ 24  11   7 903   3   0  48   0   4   0]
 [  5   0 154  96 568

## 7. Inspect MLflow runs

In [9]:
print("MLflow tracking URI:", mlflow.get_tracking_uri())
experiment = mlflow.get_experiment_by_name("assignment27-efficient-deep-learning")
if experiment is not None:
    runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    display(runs_df[[
        c for c in runs_df.columns
        if c == "run_id" or c.startswith("tags.mlflow.runName")
        or c.startswith("params.") or c.startswith("metrics.")
    ]].head(20))
else:
    print("Experiment not found. Run the training cells first.")


MLflow tracking URI: sqlite:///C:/Users/visit/Desktop/NUS/Week27/Assignment/submit/mlflow.db


,run_id,metrics.student_parameter_count,metrics.student_test_accuracy,metrics.teacher_keras_model_size_mb,metrics.student_keras_model_size_mb,metrics.teacher_parameter_count,metrics.teacher_latency_ms_per_image,metrics.teacher_test_accuracy,metrics.student_latency_ms_per_image,metrics.keras_model_size_mb,...,params.seed,params.framework,params.temperature,params.batch_size,params.architecture,params.alpha_hard_label_loss,params.learning_rate,params.dataset,params.width,tags.mlflow.runName
0,cdabde25642046869dd212a3a245420f,1898.0,0.8048,0.111071,0.036763,21386.0,9.291437,0.8517,8.531247,NaN,...,None,None,None,None,None,None,None,None,None,comparison_summary
1,b19f40fac4b84401903372905d6509f7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.036763,...,42,TensorFlow/Keras,3.0,64,SmallCNN,0.5,0.001,fashion_mnist_28x28,8,student_distillation
2,3b777546d6f9465b83f1d29db30b1c28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.111071,...,42,TensorFlow/Keras,None,64,SmallCNN,None,0.001,fashion_mnist_28x28,32,teacher_supervised
